# 01. Exploratory Data Analysis

Quick look at the turbofan engine dataset before any modelling: shape, missing
values, sensor distributions, train/test consistency and degradation trends.

Dataset summary:
- Training cycles: 20,631 rows across 100 engines
- Test engines: 100 (one current snapshot each)
- Features: cycle counter + 4 sensors (s1, s2, s3, s4). selected upstream
- Targets: `ttf` (training) and `PM_truth.txt` (test ground truth)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import load_data, SENSOR_COLS, FAULT_THRESHOLD

train, test, y_true_test = load_data('../data')
print('train:', train.shape)
print('test :', test.shape)
print('truth:', y_true_test.shape)

In [ ]:
train.head()

In [ ]:
# Per-sensor summary statistics. quick check for outliers and missing values
summary = train[SENSOR_COLS].agg(['mean', 'std', 'min', 'max']).T
summary['missing'] = train[SENSOR_COLS].isna().sum().values
summary.round(3)

No missing values across the selected sensors, so no imputation step is
needed downstream.

## Class distribution

An engine is treated as *faulty* when its remaining TTF is at most 30 cycles.
Most operating cycles are far from end-of-life, so the labels are imbalanced, which this drives the recall-focused threshold tuning in notebook 03.

In [ ]:
labels = (train['ttf'] <= FAULT_THRESHOLD).astype(int)
print(labels.value_counts())
print(f'Faulty fraction: {labels.mean():.3f}')

fig, ax = plt.subplots(figsize=(5, 3))
labels.value_counts().rename({0: 'Healthy', 1: 'Faulty'}).plot.bar(
    ax=ax, color=['#6fa8dc', '#e06666']
)
ax.set_ylabel('Cycles in training data')
ax.set_title('Class distribution (TTF threshold = 30 cycles)')
plt.tight_layout()
plt.savefig('../figures/01_class_distribution.png', dpi=150)
plt.show()

## Train vs. test distribution

Sanity check that the test snapshots come from the same operating regime as
the training cycles.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, col in zip(axes, SENSOR_COLS):
    ax.hist(train[col], bins=40, alpha=0.6, label='train',
            color='#4a86e8', density=True)
    ax.hist(test[col],  bins=40, alpha=0.6, label='test',
            color='#e06666', density=True)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Sensor feature distributions. train vs. test')
plt.tight_layout()
plt.savefig('../figures/02_train_test_distributions.png', dpi=150)
plt.show()

## Sensor trends over engine cycle

Plotting raw (unsmoothed) sensor values across all engines shows whether each
sensor carries usable degradation information.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), SENSOR_COLS):
    for eng_id, grp in train.groupby('id'):
        ax.plot(grp['cycle'], grp[col], color='steelblue',
                alpha=0.15, linewidth=0.6)
    ax.set_xlabel('Cycle')
    ax.set_ylabel(col)
    ax.set_title(f'{col} over engine life')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/03_sensor_trends.png', dpi=150)
plt.show()

Sensors `s1` and `s3` trend up with cycle count; `s2` and `s4` trend down.
All four carry monotone degradation signals despite the noise.

## Correlation heatmap

In [ ]:
cols = ['cycle'] + SENSOR_COLS
corr = train[cols].corr()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black',
                fontsize=8)
plt.colorbar(im, fraction=0.046)
ax.set_title('Feature correlations')
plt.tight_layout()
plt.savefig('../figures/04_correlation_heatmap.png', dpi=150)
plt.show()

Strong positive correlation between `s1` and `s3`; strong negative between
`s1`/`s2`, `s1`/`s4` and `s2`/`s3`. These pairs respond in opposite directions
to degradation but the redundancy isn't strong enough to drop a feature.

## EDA summary

- No missing sensor values
- Strong class imbalance. handled later with recall-targeted thresholds
- Train/test sensor distributions overlap, so the test set is representative
- Clear degradation trends in all four sensors

Proceed to notebook 02 (regression) and 03 (classification).